# CardiLearn Step 3 — SRA raw-count rescue

This notebook runs the locked raw-read rescue pipeline for `GSE232259`, `GSE52313`, `GSE186875`, and `GSE308783`.

Use a high-RAM Colab runtime or a cloud VM. STAR indexing/alignment are CPU/RAM/storage workloads; the T4 GPU is not used.

In [ ]:
%%bash
set -euo pipefail
cd /content/CardiLearn
if ! command -v micromamba >/dev/null 2>&1; then
  mkdir -p /content/bin
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/bin --strip-components=1 bin/micromamba
  chmod +x /content/bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/micromamba
eval "$(/content/bin/micromamba shell hook -s bash)"
if ! micromamba env list | grep -q 'virelion-cardi-learn-sra-v1'; then
  micromamba create -y -f envs/sra_reprocessing_v1.yml
fi
micromamba run -n virelion-cardi-learn-sra-v1 python -m py_compile scripts/step3_sra_reprocess.py
micromamba run -n virelion-cardi-learn-sra-v1 fastp --version
micromamba run -n virelion-cardi-learn-sra-v1 STAR --version
micromamba run -n virelion-cardi-learn-sra-v1 featureCounts -v


In [ ]:
import os, shutil, subprocess
print('CPU:', os.cpu_count())
print('Disk free (GB):', round(shutil.disk_usage('/content').free / 1e9, 1))
print(subprocess.run(['python','-c','import platform; print(platform.platform())'], capture_output=True, text=True).stdout)

# For the complete alignment workflow, use a high-RAM runtime or cloud VM.


## Optional tool installation

The pinned environment is in `envs/sra_reprocessing_v1.yml`. On a Linux VM with conda/mamba, install it with:

```bash
mamba env create -f envs/sra_reprocessing_v1.yml
mamba run -n virelion-cardi-learn-sra-v1 fastp --version
mamba run -n virelion-cardi-learn-sra-v1 STAR --version
mamba run -n virelion-cardi-learn-sra-v1 featureCounts -v
```

For Colab, install the binary tools with your preferred conda/micromamba bootstrap, then use the same commands below.

In [ ]:
%%bash
set -euo pipefail
cd /content/CardiLearn
export MAMBA_ROOT_PREFIX=/content/micromamba
eval "$(/content/bin/micromamba shell hook -s bash)"
micromamba activate virelion-cardi-learn-sra-v1
WORK=/content/step3_reprocessed/GSE232259
python scripts/step3_sra_reprocess.py \
  --step resolve \
  --accessions GSE232259 \
  --work-dir "$WORK"

echo 'Resolved run manifest:'
python - <<'PY'
import csv
from pathlib import Path
p=Path('/content/step3_reprocessed/GSE232259/metadata/locked_run_manifest.tsv')
rows=list(csv.DictReader(p.open(), delimiter='\t'))
print('FASTQ rows:', len(rows))
print('GSMs:', len({r['gsm'] for r in rows}))
print('SRRs:', len({r['srr'] for r in rows}))
print('Layouts:', sorted({r['library_layout'] for r in rows}))
PY

In [ ]:
%%bash
set -euo pipefail
cd /content/CardiLearn
export MAMBA_ROOT_PREFIX=/content/micromamba
eval "$(/content/bin/micromamba shell hook -s bash)"
micromamba activate virelion-cardi-learn-sra-v1
python scripts/step3_sra_reprocess.py \
  --step all \
  --accessions GSE232259 \
  --work-dir /content/step3_reprocessed/GSE232259 \
  --threads 8


In [ ]:
%%bash
set -euo pipefail
cd /content/CardiLearn
export MAMBA_ROOT_PREFIX=/content/micromamba
eval "$(/content/bin/micromamba shell hook -s bash)"
micromamba activate virelion-cardi-learn-sra-v1
for ACC in GSE52313 GSE186875 GSE308783; do
  python scripts/step3_sra_reprocess.py \
    --step all \
    --accessions "$ACC" \
    --work-dir "/content/step3_reprocessed/$ACC" \
    --threads 8
done


In [ ]:
%%bash
set -euo pipefail
cd /content/CardiLearn
export MAMBA_ROOT_PREFIX=/content/micromamba
eval "$(/content/bin/micromamba shell hook -s bash)"
micromamba activate virelion-cardi-learn-sra-v1
for ACC in GSE232259 GSE52313 GSE186875 GSE308783; do
  python scripts/step3_sra_reprocess.py \
    --step validate \
    --accessions "$ACC" \
    --work-dir "/content/step3_reprocessed/$ACC"
done
echo 'Generated source registries:'
find /content/step3_reprocessed -name external_count_sources.json -print


In [ ]:
%%bash
set -euo pipefail
cd /content/CardiLearn
export MAMBA_ROOT_PREFIX=/content/micromamba
eval "$(/content/bin/micromamba shell hook -s bash)"
micromamba activate virelion-cardi-learn-sra-v1
# Point Step 3 at one cohort at a time; do not overwrite the frozen benchmark config.
export STEP3_EXTERNAL_SOURCES=/content/step3_reprocessed/GSE232259/external_count_sources.json
python scripts/step3_baseline_runner.py


### Important

The notebook does **not** upload raw FASTQ files, BAMs, or large matrices to GitHub. Keep them in the compute workspace or object storage. Commit only small provenance/validation artifacts that you intentionally choose to publish.

Run each accession in its own work directory to cap storage use. The pipeline is resumable, so interrupted downloads, QC, alignment, and counting stages can be restarted without rebuilding completed artifacts.